# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
import csv


In [7]:
citation_header = rddCitations.first()
patent_header = rddPatents.first()

citations_no_header = rddCitations.filter(
    lambda line: line != citation_header
)

patents_no_header = rddPatents.filter(
    lambda line: line != patent_header
)

In [8]:
citations_no_header.take(3)

['3858241,956203', '3858241,1324234', '3858241,3398406']

In [9]:
patents_no_header.take(2)

['3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,']

In [10]:
citation_pairs = citations_no_header.map(
    lambda line: next(csv.reader([line]))
).map(
    lambda row: (int(row[1]), int(row[0]))
)

In [11]:
citation_pairs.take(5)

[(956203, 3858241),
 (1324234, 3858241),
 (3398406, 3858241),
 (3557384, 3858241),
 (3634889, 3858241)]

In [12]:
patent_state_pairs = patents_no_header.map(
    lambda line: next(csv.reader([line]))
).map(
    lambda row: (
        int(row[0]),
        row[5] if row[5] != "" else None
    )
)

In [13]:
patent_state_pairs.take(5)

[(3070801, None),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

In [14]:
cited_join = citation_pairs.join(
    patent_state_pairs
).cache()

In [15]:
cited_join.take(5)

[(3326372, (3861529, 'PA')),
 (3326372, (4285430, 'PA')),
 (3326372, (4537011, 'PA')),
 (3326372, (5715945, 'PA')),
 (3326372, (5937618, 'PA'))]

In [16]:
by_citing = cited_join.map(
    lambda x: (
        x[1][0],
        (x[0], x[1][1])
    )
)

In [17]:
by_citing.take(5)

[(3861529, (3326372, 'PA')),
 (4285430, (3326372, 'PA')),
 (4537011, (3326372, 'PA')),
 (5715945, (3326372, 'PA')),
 (5937618, (3326372, 'PA'))]

In [18]:
both_states = by_citing.join(
    patent_state_pairs
).cache()

In [19]:
both_states.take(5)

[(5792373, ((3649754, 'CA'), 'MI')),
 (5792373, ((3980560, 'NJ'), 'MI')),
 (5792373, ((3193100, 'NY'), 'MI')),
 (5792373, ((4642188, 'MI'), 'MI')),
 (5792373, ((4664798, 'WY'), 'MI'))]

In [20]:
same_state_rdd = both_states.filter(
    lambda x:
        x[1][0][1] is not None and
        x[1][1] is not None and
        x[1][0][1] == x[1][1]
)

In [21]:
same_state_rdd.take(10)

[(5792373, ((4642188, 'MI'), 'MI')),
 (5792373, ((5084176, 'MI'), 'MI')),
 (5792373, ((4059518, 'MI'), 'MI')),
 (5792373, ((4192750, 'MI'), 'MI')),
 (5792373, ((4836922, 'MI'), 'MI')),
 (5792373, ((4769136, 'MI'), 'MI')),
 (5792373, ((3692178, 'MI'), 'MI')),
 (5792373, ((3387712, 'MI'), 'MI')),
 (5792373, ((3703465, 'MI'), 'MI')),
 (5792373, ((3976577, 'MI'), 'MI'))]

In [22]:
same_state_ones = same_state_rdd.map(
    lambda x: (x[0], 1)
)

In [23]:
same_state_ones.take(10)

[(5792373, 1),
 (5792373, 1),
 (5792373, 1),
 (5792373, 1),
 (5792373, 1),
 (5792373, 1),
 (5792373, 1),
 (5792373, 1),
 (5792373, 1),
 (5792373, 1)]

In [24]:
same_state_counts_rdd = same_state_ones.reduceByKey(
    lambda a, b: a + b
).cache()

In [25]:
same_state_counts_rdd.take(10)

[(5792373, 11),
 (5875110, 50),
 (5565228, 2),
 (4507779, 2),
 (5172204, 1),
 (5071686, 2),
 (5727891, 2),
 (4884843, 1),
 (4639530, 14),
 (4155291, 1)]

In [26]:
top10_counts = same_state_counts_rdd.takeOrdered(
    10,
    key=lambda x: -x[1]
)

top10_counts

[(5959466, 125),
 (5983822, 103),
 (6008204, 100),
 (5952345, 98),
 (5958954, 96),
 (5998655, 96),
 (5936426, 94),
 (5913855, 90),
 (5925042, 90),
 (5951547, 90)]

In [27]:
patent_rows_rdd = patents_no_header.map(
    lambda line: next(csv.reader([line]))
)

patent_records_rdd = patent_rows_rdd.map(
    lambda row: (int(row[0]), row)
)

In [28]:
patent_records_rdd.take(2)

[(3070801,
  ['3070801',
   '1963',
   '1096',
   '',
   'BE',
   '',
   '',
   '1',
   '',
   '269',
   '6',
   '69',
   '',
   '1',
   '',
   '0',
   '',
   '',
   '',
   '',
   '',
   '',
   '']),
 (3070802,
  ['3070802',
   '1963',
   '1096',
   '',
   'US',
   'TX',
   '',
   '1',
   '',
   '2',
   '6',
   '63',
   '',
   '0',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   ''])]

In [29]:
augmented_rdd = patent_records_rdd.leftOuterJoin(
    same_state_counts_rdd
)

In [30]:
augmented_rows_rdd = augmented_rdd.map(
    lambda x: x[1][0] + [
        x[1][1] if x[1][1] is not None else 0
    ]
)

In [31]:
augmented_rows_rdd.take(2)

[['3071300',
  '1963',
  '1096',
  '',
  'US',
  'WI',
  '',
  '2',
  '',
  '226',
  '5',
  '51',
  '',
  '1',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3087568',
  '1963',
  '1215',
  '',
  'US',
  'MA',
  '',
  '2',
  '',
  '181',
  '6',
  '69',
  '',
  '3',
  '',
  '0.6667',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0]]

In [32]:
top10_rdd = augmented_rows_rdd.takeOrdered(
    10,
    key=lambda row: -row[-1]
)

In [33]:
for row in top10_rdd:
    print(row)

['5959466', '1999', '14515', '1997', 'US', 'CA', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125]
['5983822', '1999', '14564', '1998', 'US', 'TX', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103]
['6008204', '1999', '14606', '1998', 'US', 'CA', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100]
['5952345', '1999', '14501', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98]
['5958954', '1999', '14515', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96]
['5998655', '1999', '14585', '1998', 'US', 'CA', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96]
['5936426', '1999', '14466', '1997', 'US', 'CA', '5310', '2', '', '326', 

In [34]:
top10_counts


[(5959466, 125),
 (5983822, 103),
 (6008204, 100),
 (5952345, 98),
 (5958954, 96),
 (5998655, 96),
 (5936426, 94),
 (5913855, 90),
 (5925042, 90),
 (5951547, 90)]

In [35]:
patent_rows_rdd = patents_no_header.map(
    lambda line: next(csv.reader([line]))
)

patent_records_rdd = patent_rows_rdd.map(
    lambda row: (int(row[0]), row)
)

In [36]:
patent_records_rdd.take(2)

[(3070801,
  ['3070801',
   '1963',
   '1096',
   '',
   'BE',
   '',
   '',
   '1',
   '',
   '269',
   '6',
   '69',
   '',
   '1',
   '',
   '0',
   '',
   '',
   '',
   '',
   '',
   '',
   '']),
 (3070802,
  ['3070802',
   '1963',
   '1096',
   '',
   'US',
   'TX',
   '',
   '1',
   '',
   '2',
   '6',
   '63',
   '',
   '0',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   ''])]

In [37]:
augmented_rdd = patent_records_rdd.leftOuterJoin(
    same_state_counts_rdd
)

In [38]:
augmented_rows_rdd = augmented_rdd.map(
    lambda x: x[1][0] + [
        x[1][1] if x[1][1] is not None else 0
    ]
)

In [39]:
augmented_rows_rdd.take(2)

[['3070956',
  '1963',
  '1096',
  '',
  'US',
  'IN',
  '',
  '2',
  '',
  '239',
  '5',
  '59',
  '',
  '1',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0],
 ['3071496',
  '1963',
  '1096',
  '',
  'US',
  'PA',
  '',
  '2',
  '',
  '427',
  '1',
  '12',
  '',
  '3',
  '',
  '0.6667',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  0]]

In [40]:
top10_rdd = augmented_rows_rdd.takeOrdered(
    10,
    key=lambda row: -row[-1]
)

In [41]:
for row in top10_rdd:
    print(row)

['5959466', '1999', '14515', '1997', 'US', 'CA', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125]
['5983822', '1999', '14564', '1998', 'US', 'TX', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103]
['6008204', '1999', '14606', '1998', 'US', 'CA', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100]
['5952345', '1999', '14501', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98]
['5958954', '1999', '14515', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96]
['5998655', '1999', '14585', '1998', 'US', 'CA', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96]
['5936426', '1999', '14466', '1997', 'US', 'CA', '5310', '2', '', '326', 